In [1]:
import pandas as pd
import requests
import io
import os

print("🤖 A iniciar o Robô de Atualização (Pipeline ETL)...")

# --- 1. CONFIGURAÇÕES ---
# O link direto para o CSV da época atual de Portugal no football-data.co.uk
URL_DADOS_NOVOS = "https://www.football-data.co.uk/mmz4281/2324/P1.csv" # Nota: Ajusta '2324' para a época atual se necessário
CAMINHO_MESTRE = "data/P1_Mestre_5epocas.csv"

# --- 2. EXTRACT (Sacanar os dados da Web) ---
print("🌐 A ligar aos servidores da web para procurar novos jogos...")
resposta = requests.get(URL_DADOS_NOVOS)

if resposta.status_code == 200:
    # Ler o CSV da internet diretamente para a memória do pandas
    df_internet = pd.read_csv(io.StringIO(resposta.text))
    print(f"✅ Dados transferidos com sucesso! ({len(df_internet)} jogos encontrados na época atual)")
else:
    print(f"❌ Erro na ligação. Código HTTP: {resposta.status_code}")
    exit()

# --- 3. TRANSFORM (Limpar e Comparar) ---
print("🔍 A verificar a base de dados mestre...")
# Carregar a tua base de dados mestre
df_mestre = pd.read_csv(CAMINHO_MESTRE)

# Converter as datas para o mesmo formato (crucial para o pandas não se confundir)
df_mestre['Date'] = pd.to_datetime(df_mestre['Date'], format='mixed', dayfirst=True)
df_internet['Date'] = pd.to_datetime(df_internet['Date'], format='mixed', dayfirst=True)

# Descobrir qual é a data do último jogo que tu tens no teu ficheiro
ultima_data_mestre = df_mestre['Date'].max()
print(f"📅 O teu último jogo registado é de: {ultima_data_mestre.date()}")

# Filtrar a tabela da internet: Só queremos os jogos cuja data seja MAIOR que a nossa última data
df_novos_jogos = df_internet[df_internet['Date'] > ultima_data_mestre].copy()

# --- 4. LOAD (Carregar e Guardar) ---
if len(df_novos_jogos) > 0:
    print(f"⚡ Encontrados {len(df_novos_jogos)} JOGOS NOVOS! A injetar no ficheiro mestre...")
    
    # Garantir que só puxamos as colunas que o mestre tem
    colunas_em_comum = df_mestre.columns.intersection(df_novos_jogos.columns)
    df_novos_jogos = df_novos_jogos[colunas_em_comum]
    
    # Juntar o ficheiro velho com os jogos novos
    df_mestre_atualizado = pd.concat([df_mestre, df_novos_jogos], ignore_index=True)
    
    # Guardar por cima do teu ficheiro antigo (Opcional: Fazer um backup antes)
    df_mestre_atualizado.to_csv(CAMINHO_MESTRE, index=False)
    
    print("🎯 Atualização concluída com sucesso. O teu modelo tem agora os dados mais recentes!")
else:
    print("😴 Não há jogos novos na internet. O teu ficheiro mestre já está 100% atualizado.")

/Users/guilherme/Desktop/DEV/Football_ai_Predictor/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


🤖 A iniciar o Robô de Atualização (Pipeline ETL)...
🌐 A ligar aos servidores da web para procurar novos jogos...
✅ Dados transferidos com sucesso! (306 jogos encontrados na época atual)
🔍 A verificar a base de dados mestre...
📅 O teu último jogo registado é de: 2026-05-11
😴 Não há jogos novos na internet. O teu ficheiro mestre já está 100% atualizado.
